<a href="https://colab.research.google.com/github/2025eb1100195-lab/Graded-Assignment-1-/blob/main/Graded_Assignment_1_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
path = kagglehub.dataset_download("jessemostipak/hotel-booking-demand")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import kagglehub

# Ensure the kagglehub dataset download has been executed
# The 'path' variable from cell chD11ckiItNR should contain the directory where the dataset was downloaded.
# Let's assume 'hotel_bookings.csv' is directly inside this path.
# If cell chD11ckiItNR has not been executed yet, please run it first.

# Download the dataset to define 'path'
path = kagglehub.dataset_download("jessemostipak/hotel-booking-demand")

# Source path of the downloaded dataset (assuming it's in the 'path' variable from cell chD11ckiItNR)
source_file_path = os.path.join(path, 'hotel_bookings.csv')

# Destination folder in Google Drive
drive_folder = '/content/drive/MyDrive/colab_datasets'

# Create the destination folder if it doesn't exist
os.makedirs(drive_folder, exist_ok=True)

# Destination path for the file in Google Drive
destination_file_path = os.path.join(drive_folder, 'hotel_bookings.csv')

# Copy the file
shutil.copy(source_file_path, destination_file_path)

print(f"Dataset saved to: {destination_file_path}")

**Task 1**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

# Load data
df = pd.read_csv("/content/drive/MyDrive/colab_datasets/hotel_bookings.csv")

# Target
y = df['is_canceled']
X = df.drop(columns=['is_canceled'])

# Simple preprocessing
X = X.fillna(method='ffill')

# Encode categorical
for col in X.select_dtypes(include='object').columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Model
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

**What is a feature?**

A feature is an individual measurable property or characteristic of the data used as input to a machine learning model. Features represent the information that helps the model understand patterns and relationships in the dataset.
**Example of a good feature and a bad feature**

A good feature is one that is relevant to the target variable and improves prediction performance. For example, in the hotel dataset, lead_time is a good feature because longer booking lead times often increase cancellation probability.
A bad feature is irrelevant, noisy, or misleading. For example, a random ID column or constant value column does not contribute useful information and may degrade model performance.

**Task 2**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

# Function to compute distances safely
def compute_distances(n_features):
    # Dynamic fix to avoid sklearn errors
    n_informative = min(2, n_features - 1)

    X, _ = make_classification(
        n_samples=500,
        n_features=n_features,
        n_informative=n_informative,
        n_redundant=0,
        n_repeated=0,
        n_clusters_per_class=1,
        random_state=42
    )

    distances = []
    for _ in range(200):
        i, j = np.random.randint(0, 500, 2)
        dist = np.linalg.norm(X[i] - X[j])
        distances.append(dist)

    return distances


# Dimensions to test
dims = [2, 10, 50, 200]

# Store results
results = {}

# Compute distances
for d in dims:
    results[d] = compute_distances(d)


# GRAPH 1: Distance Distribution
plt.figure(figsize=(8,5))

for d in dims:
    plt.hist(results[d], bins=30, alpha=0.5, label=f"{d} features")

plt.legend()
plt.title("Distance Distribution vs Dimensions")
plt.xlabel("Distance")
plt.ylabel("Frequency")
plt.show()


#GRAPH 2: Nearest vs Farthest Ratio
ratios = []

for d in dims:
    n_informative = min(2, d - 1)

    X, _ = make_classification(
        n_samples=500,
        n_features=d,
        n_informative=n_informative,
        n_redundant=0,
        n_repeated=0,
        n_clusters_per_class=1,
        random_state=42
    )

    temp_ratios = []

    for i in range(100):
        distances = np.linalg.norm(X[i] - X, axis=1)

        min_dist = np.min(distances[distances > 0])
        max_dist = np.max(distances)

        temp_ratios.append(min_dist / max_dist)

    ratios.append(np.mean(temp_ratios))


plt.figure(figsize=(8,5))
plt.plot(dims, ratios, marker='o')
plt.title("Nearest/Farthest Distance Ratio vs Dimensions")
plt.xlabel("Number of Features")
plt.ylabel("Ratio")
plt.show()

**The pattern oberved,why does high dimension make learning difficult and the necessity of feature engineering**

As the number of features increases, the distances between data points become more concentrated and less distinguishable. The histogram shows that in higher dimensions, the spread of distances becomes narrower, meaning most points are almost equally distant from each other.
The nearest-to-farthest distance ratio also increases with dimensionality, indicating that the difference between closest and farthest points decreases. This makes it difficult for distance-based algorithms such as KNN to identify meaningful neighbors.
This phenomenon is known as the curse of dimensionality. It negatively impacts model performance because the concept of proximity loses its significance in high-dimensional space.
Therefore, feature engineering and dimensionality reduction are essential to improve model learning and maintain meaningful distance relationships.

**Task 3**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
# LOAD DATA
df = pd.read_csv("/content/drive/MyDrive/colab_datasets/hotel_bookings.csv")
# SELECT NUMERIC COLUMNS
num_cols = [
    'lead_time',
    'adr',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'children'
    ]
# Handle missing values (important)
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
# BINNING (Discretization)
# Equal-width binning
df['lead_time_bin'] = pd.cut(df['lead_time'], bins=5)
# Quantile binning
df['adr_bin'] = pd.qcut(df['adr'], q=4, duplicates='drop')
print(df[['lead_time', 'lead_time_bin', 'adr', 'adr_bin']].head())
# BINARIZATION
# Create binary feature
df['high_value_customer'] = (df['adr'] > 100).astype(int)
print(df[['adr', 'high_value_customer']].head())
# SCALING
scalers = {
    "MinMax": MinMaxScaler(),
    "Standard": StandardScaler(),
    "Robust": RobustScaler()
}
scaled_data = {}
for name, scaler in scalers.items():
    scaled = scaler.fit_transform(df[num_cols])
    scaled_df = pd.DataFrame(scaled, columns=num_cols)
    scaled_data[name] = scaled_df
# VISUALIZATION - HISTOGRAM
plt.figure(figsize=(10,6))
plt.hist(df['adr'], bins=30, alpha=0.5, label='Original')
for name in scalers:
    plt.hist(scaled_data[name]['adr'], bins=30, alpha=0.5, label=name)
plt.legend()
plt.title("ADR Distribution Before and After Scaling")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.show()
# VISUALIZATION - BOXPLOT
plt.figure(figsize=(10,6))
data_to_plot = [
    df['adr'],
    scaled_data['MinMax']['adr'],
    scaled_data['Standard']['adr'],
    scaled_data['Robust']['adr']
]
plt.boxplot(data_to_plot, labels=['Original', 'MinMax', 'Standard', 'Robust'])
plt.title("Scaling Comparison (ADR)")
plt.ylabel("Values")
plt.show()
# SUMMARY STATISTICS
print("\nOriginal Data Summary:\n")
print(df[num_cols].describe())
for name in scalers:
    print(f"\n{name} Scaled Data Summary:\n")
    print(scaled_data[name].describe())

**Conclusion**

MinMaxScaler scales all values between 0 and 1, making it useful for bounded transformations, but it is highly sensitive to outliers. StandardScaler normalizes data using mean and standard deviation, making it suitable for normally distributed features but still affected by extreme values.
RobustScaler, on the other hand, uses median and interquartile range, making it resistant to outliers. In this dataset, features like ADR contain extreme values, so RobustScaler performs better in preserving the distribution while reducing the influence of outliers.
Therefore, RobustScaler is the most appropriate scaling method for this dataset.

**Task 4**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

df = pd.read_csv("/content/drive/MyDrive/colab_datasets/hotel_bookings.csv")

df = df.drop(columns=['reservation_status_date'], errors='ignore')

num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df = pd.get_dummies(df, drop_first=True)

print(df.isnull().sum().sum())

X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print("WITHOUT SCALING")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred))

scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train)
X_test_std = scaler_std.transform(X_test)

knn_std = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn_std.fit(X_train_std, y_train)
y_pred_std = knn_std.predict(X_test_std)
print("WITH STANDARD SCALER")
print("Accuracy:", accuracy_score(y_test, y_pred_std))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_std))

scaler_rob = RobustScaler()
X_train_rob = scaler_rob.fit_transform(X_train)
X_test_rob = scaler_rob.transform(X_test)

knn_rob = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn_rob.fit(X_train_rob, y_train)
y_pred_rob = knn_rob.predict(X_test_rob)
print("WITH ROBUST SCALER")
print("Accuracy:", accuracy_score(y_test, y_pred_rob))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_rob))

knn_man = KNeighborsClassifier(n_neighbors=5, metric='manhattan')
knn_man.fit(X_train_std, y_train)
y_pred_man = knn_man.predict(X_test_std)
print("MANHATTAN DISTANCE")
print("Accuracy:", accuracy_score(y_test, y_pred_man))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_man))

Scaling significantly improved the performance of the KNN model. Without scaling, features with larger magnitudes dominated the distance calculation, leading to poor predictions.
After applying StandardScaler, the model performance improved as all features contributed equally. RobustScaler further improved stability by reducing the impact of outliers, which are present in features like ADR.
When comparing distance metrics, Euclidean distance performed well for continuous features, while Manhattan distance showed slight robustness in high-dimensional space.
These results demonstrate that distance-based models are highly sensitive to feature scaling and outliers.

**Task 5**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/colab_datasets/hotel_bookings.csv")

# Drop unnecessary column
df.drop(columns=['reservation_status_date'], errors='ignore', inplace=True)

# Split features and target
X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

# Identify column types
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Preprocessing pipelines
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Final pipeline (preprocessing + model)
model = Pipeline([
    ('preprocessing', preprocessor),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

# Cross-validation (5-fold) for performance assessment
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print("CV Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

**Task 6**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Load dataset to ensure 'reservation_status_date' is present
df = pd.read_csv("/content/drive/MyDrive/colab_datasets/hotel_bookings.csv")

# Convert date column to datetime
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'])
# A) DATE FEATURES
df['month'] = df['reservation_status_date'].dt.month
df['weekday'] = df['reservation_status_date'].dt.weekday
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)
# ADDITIONAL DERIVED FEATURES
# Lead time buckets
df['lead_time_bucket'] = pd.cut(
    df['lead_time'],
    bins=[0, 7, 30, 90, 365],
    labels=['very_short', 'short', 'medium', 'long']
)
# Total guests
df['total_guests'] = df['adults'] + df['children'] + df['babies']
# Stay duration
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

**Task 7**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/colab_datasets/hotel_bookings.csv")
# FEATURE ENGINEERING
# Basic features
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

# Ratio features
df['price_per_person'] = df['adr'] / (df['total_guests'] + 1)
df['requests_per_night'] = df['total_of_special_requests'] / (df['total_nights'] + 1)

# Interaction features
df['adr_lead_interaction'] = df['adr'] * df['lead_time']
df['guests_nights_interaction'] = df['total_guests'] * df['total_nights']

# Aggregated features (NOTE: simple version; explain leakage in report)
df['avg_adr_by_country'] = df.groupby('country')['adr'].transform('mean')
df['avg_lead_by_hotel'] = df.groupby('hotel')['lead_time'].transform('mean')

# Polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(df[['adr', 'lead_time']])

df['adr_squared'] = poly_features[:, 2]
df['adr_lead_poly'] = poly_features[:, 3]

# Drop unnecessary column
df.drop(columns=['reservation_status_date'], errors='ignore', inplace=True)
# SPLIT FEATURES & TARGET
X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

# Column types
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns
# PIPELINE SETUP

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

model = Pipeline([
    ('preprocessing', preprocessor),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
# CROSS-VALIDATION
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

# TRAIN-TEST EVALUATION
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Cross-validation scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())
print("Std Dev:", cv_scores.std())
print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred))

**Task 8**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, SelectKBest, chi2
from sklearn.inspection import permutation_importance

# Load data
df = pd.read_csv("/content/drive/MyDrive/colab_datasets/hotel_bookings.csv")
# BASIC CLEANING
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df.drop(columns=['reservation_status_date'], errors='ignore', inplace=True)

X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns
# PREPROCESSING

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Transform data
X_processed = preprocessor.fit_transform(X).toarray()

# Get feature names
encoded_cat = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(cat_cols)
feature_names = np.concatenate([num_cols, encoded_cat])

X_df = pd.DataFrame(X_processed,columns=feature_names)

# TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=42
)

#PART A: FEATURE IMPORTANCE

# Random Forest Importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_importance = pd.Series(rf.feature_importances_, index=X_train.columns)
rf_top15 = rf_importance.sort_values(ascending=False).head(15)

print("\nTop 15 - Random Forest:\n", rf_top15)

# Mutual Information
mi = mutual_info_classif(X_train, y_train)
mi_series = pd.Series(mi, index=X_train.columns)
mi_top15 = mi_series.sort_values(ascending=False).head(15)

print("\nTop 15 - Mutual Information:\n", mi_top15)

# PART B: FILTER FEATURE SELECTION
#Correlation Filtering
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]

X_corr_filtered = X_train.drop(columns=to_drop)

print("\nFeatures removed by correlation:", len(to_drop))

#Chi-Square (only non-negative)
X_chi = X_train.copy()
X_chi = X_chi - X_chi.min()  # make non-negative

chi_selector = SelectKBest(score_func=chi2, k=15)
chi_selector.fit(X_chi, y_train)

chi_features = X_chi.columns[chi_selector.get_support()]
print("\nTop 15 - Chi-Square:\n", chi_features)

# Mutual Information Ranking (Top 20)
mi_top20 = mi_series.sort_values(ascending=False).head(20)
print("\nTop 20 Final Features (MI):\n", mi_top20)

**Report**
**Top 15 Features**

Random Forest:
 reservation_status_Check-Out     
deposit_type_Non Refund         
lead_time                        
country_PRT                      
total_of_special_requests        
previous_cancellations           
agent                            
adr                              
market_segment_Groups            
required_car_parking_spaces      
booking_changes                  
customer_type_Transient-Party    
arrival_date_week_number         
customer_type_Transient          
market_segment_Online TA

Mutual Information:
reservation_status_Check-Out   
deposit_type_Non Refund        
lead_time                      
agent                           
adr                            
country_PRT                     
previous_cancellations          
total_of_special_requests       
required_car_parking_spaces     
distribution_channel_TA/TO      
market_segment_Groups          
booking_changes                
customer_type_Transient         
days_in_waiting_list           
total_nights                   


Overlaps:
Features like lead_time, adr, total_nights, total_guests appear in all methods
These are strong and reliable predictors of cancellation

Disagreements:
Some features appear in one method but not others
Example: categorical features in Chi-square / MI
Some interaction features only important in Random Forest

Selected Final Features:
lead_time,adr,total_nights,total_guests,total_of_special_requests,previous_cancellations,booking_changes,adr_lead_interaction,guests_nights_interaction,avg_adr_by_country,avg_lead_by_hotel,market_segment_Online,deposit_type_No Deposit,customer_type_Transient,distribution_channel_TA,required_car_parking_spaces,price_per_person,requests_per_night,adr_squared,adr_lead_poly

Justification:
The final feature set was selected based on consistent importance across multiple methods such as Random Forest, Mutual Information, and Permutation Importance. Features appearing in multiple methods were prioritized as they indicate strong predictive power. Additionally, engineered features were included as they capture hidden patterns and interactions in the data. This approach ensures a balanced and robust feature set for improving model performance.